# 🛩️ Example: Working with the Ultrastick-25e Model

This notebook demonstrates the usage of the linear longitudinal model of the Ultrastick-25e aircraft in the TensorAeroSpace environment.

## 📋 What we will do:
1. Import the required libraries
2. Configure time parameters and the reference signal
3. Create and initialize the environment
4. Execute one simulation step
5. Analyze the results

## 📚 Importing Libraries

Loading all necessary modules for working with the aircraft model:

In [ ]:
# Основные библиотеки
import gymnasium as gym
import numpy as np

# Импорт TensorAeroSpace для регистрации окружений
import tensoraerospace

# Модули TensorAeroSpace
from tensoraerospace.utils import generate_time_period, convert_tp_to_sec_tp
from tensoraerospace.signals.standart import unit_step

## ⚙️ Simulation Parameters Setup

Defining time parameters and creating a reference signal for pitch angle control:

In [ ]:
# Параметры дискретизации
dt = 0.01  # Шаг дискретизации (секунды)

# Генерация временного периода
tp = generate_time_period(tn=20, dt=dt)  # 20 секунд симуляции
tps = convert_tp_to_sec_tp(tp, dt=dt)    # Преобразование в секунды
number_time_steps = len(tp)             # Общее количество временных шагов

# Создание ступенчатого опорного сигнала для угла тангажа (theta)
reference_signals = np.reshape(
    unit_step(degree=5, tp=tp, time_step=10, output_rad=True), 
    [1, -1]
)

print(f"📊 Параметры симуляции:")
print(f"   • Время симуляции: {tp[-1]:.1f} сек")
print(f"   • Шаг дискретизации: {dt} сек")
print(f"   • Количество шагов: {number_time_steps}")
print(f"   • Форма опорного сигнала: {reference_signals.shape}")

## 🚀 Creating and Initializing the Environment

Creating the Ultrastick-25e environment with the specified parameters and performing a reset:

In [ ]:
# Создание среды Ultrastick-25e
env = gym.make(
    "LinearLongitudinalUltrastick-v0",
    number_time_steps=number_time_steps,
    initial_state=[[0], [0], [0], [0], [0]],  # [u, w, q, theta, h]
    reference_signal=reference_signals,
    tracking_states=["theta"]
)

# Инициализация среды
state, info = env.reset()

print(f"✅ Среда успешно создана и инициализирована!")
print(f"📈 Начальное состояние: {state.flatten()}")
print(f"🎯 Форма пространства состояний: {env.observation_space.shape}")
print(f"🎮 Форма пространства действий: {env.action_space.shape}")
print(f"📊 Отслеживаемые состояния: {env.unwrapped.tracking_states}")
print(f"📋 Пространство состояний: {env.unwrapped.state_space}")
print(f"📤 Пространство выходов: {env.unwrapped.output_space}")

## 🎮 Executing a Simulation Step

Applying a control action and observing the system response:

In [ ]:
# Применение управляющего воздействия
# Действие должно быть одномерным массивом
control_input = np.array([1.0], dtype=np.float32)  # Форма (1,) для среды

# Выполнение одного шага симуляции
state, reward, terminated, truncated, info = env.step(control_input)

print(f"🎯 Управляющее воздействие: {control_input[0]:.2f} градусов")
print(f"📊 Новое состояние: {state.flatten()}")
theta_idx = env.unwrapped.state_space.index("theta")
q_idx = env.unwrapped.state_space.index("q")
print(f"   • Угол тангажа (theta): {np.rad2deg(state[theta_idx, 0]):.4f} град")
print(f"   • Угловая скорость (q): {np.rad2deg(state[q_idx, 0]):.4f} град/с")
# Обработка награды (может быть массивом или скаляром)
reward_value = reward[0] if isinstance(reward, np.ndarray) else reward
print(f"🏆 Награда: {reward_value:.6f}")
print(f"🔚 Завершено: {terminated}")
print(f"⏰ Прервано: {truncated}")